# 05 - Train, test and residuals

A calibration always improves the fit on the data it was fitted to. The only
honest question is whether it also holds on data it has **not** seen.

The setup mirrors how you would do this on a real plant:

1. The record runs from 1 to 8 January. The first five days are the training
   window, the last three are held back.
2. The parameter is fitted on the training window alone.
3. The model then runs **once, continuously, over the whole record** the same
   way the plant did.
4. The held-back part of that run is compared against the measurements.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from demo_plant import DEFAULT_K_HYD_CH, TRUE_PARAMETERS, build_demo_plant, make_twin_measurements, simulate

from pyadm1ode_calibration import InitialCalibrator

SPLIT = "2024-01-06"

measurements = make_twin_measurements(days=8, noise=0.02, seed=1)
train = measurements.get_time_window("2024-01-01", SPLIT)

is_test = measurements.data.index >= SPLIT
observed = measurements.data["Q_gas"].to_numpy()

print(f"record   : {measurements.data.index[0]:%Y-%m-%d} .. {measurements.data.index[-1]:%Y-%m-%d}  ({len(observed)} rows)")
print(f"training : {(~is_test).sum()} rows up to {SPLIT}")
print(f"test     : {is_test.sum()} rows from {SPLIT} onwards")

## Calibrate on the training window only

The test window is never passed to the calibrator. That is what makes it an
independent check later on.

In [ ]:
result = InitialCalibrator(build_demo_plant(days=5), verbose=False).calibrate(
    train,
    parameters=["k_hyd_ch"],
    bounds={"k_hyd_ch": (1.0, 15.0)},
    objectives=["Q_gas"],
    method="nelder_mead",
    max_iterations=20,
    sensitivity_analysis=False,
)
print(f"calibrated k_hyd_ch = {result.parameters['k_hyd_ch']:.3f}  (true value {TRUE_PARAMETERS['k_hyd_ch']})")

## One run over the whole record

Now the model runs from 1 January to 8 January in one go. Once with the default
parameter, once with the calibrated one. Only afterwards do we cut the result
into the two windows and score them separately.

In [ ]:
plant = build_demo_plant(days=8)

runs = {
    "default": np.asarray(simulate(plant, measurements, {"k_hyd_ch": DEFAULT_K_HYD_CH})["Q_gas"], dtype=float),
    "calibrated": np.asarray(simulate(plant, measurements, result.parameters)["Q_gas"], dtype=float),
}

def metrics(simulated, mask):
    obs, sim = observed[mask], simulated[mask]
    rmse = float(np.sqrt(np.mean((sim - obs) ** 2)))
    r2 = float(1.0 - np.sum((obs - sim) ** 2) / np.sum((obs - obs.mean()) ** 2))
    bias = float(sim.mean() / obs.mean() - 1.0)
    return rmse, r2, bias

print(f"{'window':9s} {'parameters':12s} {'RMSE':>8s} {'R2':>8s} {'bias':>8s}")
for name, curve in runs.items():
    for window, mask in (("training", ~is_test), ("test", is_test)):
        rmse, r2, bias = metrics(curve, mask)
        print(f"{window:9s} {name:12s} {rmse:8.1f} {r2:8.3f} {bias:+8.2%}")

print(f"\nmeasurement noise in the test window: {observed[is_test].std():.1f} m3/d")

Three things to read out of this.

**On the training window the calibration works.** RMSE falls from 253 to 48 and
the bias from +4.5 % to +0.1 %. That is the fit the optimiser was asked for.

**On the test window the calibrated model is as good as the data allows.** Its
RMSE of about 57 m3/d equals the measurement noise of 57.5 m3/d, there is
nothing left to explain. The improvement over the default is small there (58.3
to 56.9), and that is informative rather than disappointing. By the second half
of the record the digester is close to steady state, and there the gas
production is set by what is fed in, not by how fast hydrolysis runs. What
identified `k_hyd_ch` was the start-up ramp inside the training window.

**A test window only validates what it exercises.** If you want the check to
carry weight, hold back a stretch in which the calibrated parameter actually
acts. A load change, a substrate switch, a restart.

## Seeing it

One figure, one continuous simulation, the split marked. The left part is what
the optimiser saw; everything right of the line is the check.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(measurements.data.index, observed, label="measured", alpha=0.6)
ax.plot(measurements.data.index, runs["default"], ls=":", label=f"default (k_hyd_ch={DEFAULT_K_HYD_CH})")
ax.plot(measurements.data.index, runs["calibrated"], label=f"calibrated (k_hyd_ch={result.parameters['k_hyd_ch']:.2f})")
ax.axvline(measurements.data.index[is_test][0], color="k", lw=1)
ax.text(measurements.data.index[is_test][0], ax.get_ylim()[1], "  test window ->", va="top", fontsize=10)
ax.set_ylabel("Q_gas [m3/d]")
ax.set_title("One simulation across the record, scored on the test window")
ax.legend(loc="center right")
fig.tight_layout()

## Residuals

Averages hide structure. The residual of the held-back part answers a different
question: is what remains just noise, or is the model still systematically wrong
somewhere?

In [ ]:
predicted = runs["calibrated"][is_test]
test_index = measurements.data.index[is_test]
residuals = observed[is_test] - predicted

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(test_index, residuals, ".")
axes[0].axhline(0, color="k", lw=0.8)
axes[0].set_ylabel("residual [m3/d]")
axes[0].set_title("Residuals over the test window")
axes[0].tick_params(axis="x", rotation=30)

axes[1].hist(residuals, bins=20, color="tab:blue")
axes[1].set_title("Distribution")
fig.tight_layout()

print(f"mean residual      : {residuals.mean():8.2f} m3/d   (0 would be unbiased)")
print(f"std residual       : {residuals.std():8.2f} m3/d")
print(f"measurement scatter: {observed[is_test].std():8.2f} m3/d   <- what noise alone accounts for")

### The statistical tests

`CalibrationValidator.analyze_residuals()` runs three checks. Printing the
returned object dumps every residual, so ask it the three questions instead.

In [ ]:
from pyadm1ode_calibration import CalibrationValidator

test_window = measurements.get_time_window(SPLIT, "2024-01-09")
analysis = CalibrationValidator(build_demo_plant(days=3), verbose=False).analyze_residuals(
    test_window, {"Q_gas": predicted}, objectives=["Q_gas"]
)["Q_gas"]

print(f"residuals analysed      : {len(analysis.residuals)}")
print()
print(f"normally distributed?   : {analysis.is_normally_distributed()}"
      f"   (Shapiro-Wilk p = {analysis.normality_test['p_value']:.3f})")
print(f"autocorrelated?         : {analysis.has_autocorrelation()}"
      f"   (lag-1 = {analysis.autocorrelation:+.2f})")
print(f"heteroscedastic?        : {analysis.has_heteroscedasticity()}"
      f"   (p = {analysis.heteroscedasticity_test['p_value']:.3f})")
print(f"outliers beyond 3 sigma : {analysis.outlier_indices}")

All three come back negative and the residual scatter matches the scatter of the
measurements themselves: what is left over is the noise we put into the data,
and nothing else.